# 1. Simulated Scenario
I'm an Analyst on the Customer Experience team at a Brazilian online marketplace (think: Olist, ~2018). In a leadership sync, the VP of Ops says:
"Our review scores have been softer lately, and we're getting more delivery-related complaints. Before we make any big calls on logistics spend, can you dig into what's actually going on and tell us what we should do?"

# 2. Clarifying Questions I'd Ask (I'd think through)
To find out what's actually going on and what we should do, I would first break down the problem to identify the goal or the problem we're trying to address. After defining the goal, it is also important to communicate the priority of this problem, and the actual resource (data) we have in hand. This process is important as it assures that everyone's on the same page and keep the team's work/goal consistent.
## Define The Problem  
The review scores are softer compared to what/when?   
 - Is it a sudden drop (could imply system errors or change in platform interface/policies/carriers...etc.) or a trend?
 - Is the drop in a certain group/segment/region?   
What defines a lower review score?   
 - Are we measuring by the mean, mode, or other?    
 - Are there any outliers?    
Should I be testing the assumption that delivery-related complaints are causing lower review scores?  
 - What counts as delivery-related complaints?   
 - Covariates?
## Practical Constraints
How urgent is this? 
 - How detail should i go into this problem? Should I prioritize this over other tasks I have in hand?
 - When Should the team gather and discuss again?
What decision would my analyses support? (e.g., adjusting logistics spend?)  
 - What actual options we have on the table? (e.g., change communication to customers? negotiate with carriers?)

These questions might not all have a clear/immediate answer, but thinking through it helps me map out the initial goal and analysis plan.

# 3. What's the Goal? What's the Question/Assumption to ask/test?
Since this is a simulated scenario, I'd like to look into the actual data before making any assumptions

In [ ]:
# Loading libraries
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import sys
from pathlib import Path

sys.path.append("../src")
from helpers import get_connection, sql

In [ ]:
# Loading datasets as tables into SQLite
conn = get_connection(["orders", "reviews", "order_items", "customers"])

In [7]:
# Check the time span of the dataset
sql("" \
"SELECT MIN(order_purchase_timestamp) AS earliest_order, " \
"MAX(order_purchase_timestamp) AS latest_order " \
"FROM orders;", conn)

,earliest_order,latest_order
0,2016-09-04 21:15:19,2018-10-17 17:30:18


In [ ]:
# Check the mean of review scores by month
sql("" \
"SELECT strftime('%Y-%m', review_creation_date) AS year_month, " \
"ROUND(AVG(review_score), 2) AS avg_score, " \
"COUNT(*) AS num_reviews " \
"FROM reviews " \
"GROUP BY year_month " \
"ORDER BY year_month;", conn)

,year_month,avg_score,num_reviews
0,2016-10,4.06,179
1,2016-11,3.19,101
2,2016-12,2.36,45
3,2017-01,4.34,239
4,2017-02,4.28,1413
5,2017-03,4.03,2481
6,2017-04,4.04,2055
7,2017-05,4.10,3710
8,2017-06,4.13,3440
9,2017-07,4.18,3503


## Timeframe and Baseline Assumption
Given that the VP in the simulated scenario said "the review scores have been softer lately", I'm assuing this scenario happened in early April 2018, whereas "lately" refers to Jan-Mar 2018. For the baseline, I'm assuming the review scores are softer compared to 2017's average, as it's the most stable reference point here. However, it's possible that Jan-Mar has a recurring seasonal dip that is independent of other specific issue. I will come back to this in the actual analysis.

# What data is available to support the question? (Feasibility?)

#